# Notebook 2: Pea Protein Chain — Full Rebuild

**Project:** RAW Project — 3D-Printed Biopolymer Panels  
**Database:** `lca_database_3DPrintedBiopol`  
**Ecoinvent version:** 3.12, cut-off system model  
**Date:** 2026  

## Purpose

This notebook rebuilds the pea protein supply chain from scratch and restructures the downstream mix preparation and 3D printing activities. It implements the following modelling decisions made in June 2026:

1. **Scope of `pea protein binder production`:** Option A — the activity produces only the binder paste (water + glycerol + xanthan gum + pea protein isolate). Filler addition is handled separately in `mix preparation`.
2. **AEIEP foreground process:** No suitable ecoinvent proxy exists for pea protein isolate production. A minimal foreground process is built from literature-based energy and material inputs.
3. **Recipe reference:** Recipe 1 from Eleftheria's protocol (Gitsouli, DTU/CITA) is used as the reference case. The suggested trial recipe may be added as a parameterised variant in a future notebook. Mass fractions are re-normalised to exclude the filler fraction when building binder-only activities.
4. **Water evaporation:** Not modelled during binder preparation. The protocol does not mention significant evaporation at this stage.
5. **Mix preparation as separate activity:** Filler addition and homogenisation are a distinct physical step from binder formulation, with different energy inputs and machinery. A new `mix preparation` activity is created.
6. **Kiln baking duration:** Corrected to 45 minutes at 150°C, per protocol. The previous value of 2 hours was incorrect.
7. **Calcium chloride removed:** Not present in Eleftheria's protocol. Removed from `3D printing`.
8. **Glycerol relocated:** Glycerol is a binder ingredient, not a 3D printing input. Removed from `3D printing` and added to `pea protein binder production`.
9. **All filler inputs relocated to mix preparation:** All filler exchanges are moved from `3D printing` to `mix preparation`, which now holds the complete ingredient input set. Zero-amount placeholders are retained for fillers not used in Recipe 1, to support future optimisation runs.

### Activities created or modified in this notebook

| Activity | Action | Notes |
|---|---|---|
| `AEIEP pea protein isolate production` | **Create new** | Foreground process; literature-based LCI |
| `pea protein binder production` | **Rebuild** | Was near-empty; now fully specified |
| `mix preparation` | **Create new** | New activity; receives all ingredient inputs |
| `3D printing` | **Modify** | Remove fillers, glycerol, CaCl₂; add mix input |
| `Kiln baking - 150 degress for 2 hours` | **Modify** | Correct electricity to 45 min duration |

### What is NOT changed in this notebook

- Filler production activities (bark, cellulose, cotton, hemp dust, seagrass, wood flour) — unchanged from Notebook 1
- Drying with dehumidifying chamber — unchanged
- Avoided burden activity for nitrogen fixation — deferred to a later notebook

---

## 0. Setup and sanity check

In [1]:
import bw2data as bd
import bw2io as bi

# Set the active project
bd.projects.set_current("biopol_lca")

# Open databases
db = bd.Database("lca_database_3DPrintedBiopol")
ei = bd.Database("ecoinvent-3.12-cutoff")

print(f"Foreground database: {len(db)} activities")
print()
print("Activities currently in database:")
for act in sorted(db, key=lambda x: x['name']):
    exchanges = list(act.exchanges())
    inputs = [e for e in exchanges if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(inputs)} technosphere input(s)")

Foreground database: 11 activities

Activities currently in database:
  3D printing (RER) — 11 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking - 150 degress for 2 hours (RER) — 3 technosphere input(s)
  avoided burden: nitrogen fixation, peas (GLO) — 1 technosphere input(s)
  bark flour filler production (GLO) — 4 technosphere input(s)
  cellulose filler production (GLO) — 4 technosphere input(s)
  cotton filler production (GLO) — 4 technosphere input(s)
  hemp dust filler production (GLO) — 2 technosphere input(s)
  pea protein binder production (GLO) — 1 technosphere input(s)
  seagrass filler production (GLO) — 2 technosphere input(s)
  wood flour filler production (GLO) — 4 technosphere input(s)


---

## 1. Helper functions

Two helper functions are defined here and reused throughout:

- `find_activity`: exact name match in the foreground database. Raises an error if not found or ambiguous.
- `find_ei`: look up an ecoinvent dataset by name, location, and optionally reference product.
- `search_ei`: broad fuzzy search across ecoinvent by keyword. Use this interactively to identify the correct dataset before hard-coding it in `find_ei`.
- `delete_exchange`: remove all technosphere exchanges linking a given input from an activity.
- `clear_all_technosphere`: remove every technosphere exchange from an activity (used when rebuilding from scratch).

In [2]:
def find_activity(db, name):
    """Find a single foreground activity by exact name."""
    results = [a for a in db if a['name'] == name]
    if len(results) == 0:
        raise ValueError(f"Activity not found: '{name}'")
    if len(results) > 1:
        raise ValueError(f"Multiple activities found for: '{name}'")
    return results[0]


def find_ei(name, location, ref_product=None):
    """Look up an ecoinvent activity by exact name and location."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


def search_ei(keyword, max_results=20):
    """
    Broad keyword search across ecoinvent. Use interactively to identify
    the correct dataset name before hard-coding it in find_ei().
    """
    keyword_lower = keyword.lower()
    results = [
        a for a in ei
        if keyword_lower in a['name'].lower()
    ]
    print(f"Found {len(results)} dataset(s) matching '{keyword}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print(f"  unit:     {a.get('unit', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more. Narrow your keyword.")


def delete_exchange(activity, input_activity):
    """Delete all technosphere exchanges pointing to input_activity from activity."""
    to_delete = [
        exc for exc in activity.exchanges()
        if exc['type'] == 'technosphere'
        and exc.input.key == input_activity.key
    ]
    for exc in to_delete:
        exc.delete()
    return len(to_delete)


def clear_all_technosphere(activity):
    """Remove every technosphere exchange from an activity. Used when rebuilding from scratch."""
    to_delete = [
        exc for exc in activity.exchanges()
        if exc['type'] == 'technosphere'
    ]
    for exc in to_delete:
        exc.delete()
    print(f"  Removed {len(to_delete)} technosphere exchange(s) from '{activity['name']}'")


print("Helper functions defined.")

Helper functions defined.


---

## 2. Retrieve shared background datasets

All background datasets used across this notebook are retrieved here in one place.
Use `search_ei()` interactively to verify any dataset before its `find_ei()` call if needed.

### Proxy notes

**Peas (raw material):** Ecoinvent 3.12 does not contain a food-grade pea protein isolate market. The closest available dataset is `market for peas, feed` or `market for field peas`, which represents whole dried peas at farm gate. This is used as the raw material input into AEIEP production. Flag for sensitivity analysis.

**Xanthan gum:** Search `search_ei('xanthan')` at runtime. If absent, fall back to `market for sodium carboxymethyl cellulose | RER` (CMC) as a functional analogue polysaccharide thickener. Given the very small amount (0.001 kg per kg binder), the proxy choice has negligible impact on all impact categories.

**Tap water:** Used for the AEIEP wet extraction stage. Search `search_ei('tap water')` to confirm the exact dataset name.

In [5]:
# ── Electricity ───────────────────────────────────────────────────────────────
ei_electricity = find_ei(
    "market for electricity, medium voltage", "DE",
    ref_product="electricity, medium voltage"
)

# ── Machine wear ──────────────────────────────────────────────────────────────
ei_machine = find_ei(
    "market for industrial machine, heavy, unspecified", "RER",
    ref_product="industrial machine, heavy, unspecified"
)

# ── Transport ─────────────────────────────────────────────────────────────────
ei_transport = find_ei(
    "market for transport, freight, lorry, unspecified", "RER",
    ref_product="transport, freight, lorry, diesel, unspecified"
)

# ── Glycerol ──────────────────────────────────────────────────────────────────
ei_glycerol = find_ei(
    "market for glycerine", "RER",
    ref_product="glycerine"
)

# ── Peas (raw material for AEIEP) ─────────────────────────────────────────────
# search_ei('peas')  # ← Run this cell interactively first to confirm the dataset name
# Expected candidates: 'market for peas, feed' or 'market for field peas'
# Hard-code the confirmed name below after running search_ei:
ei_peas = find_ei(
    "market for protein pea", "GLO",  # ⚠️ Confirm exact name and location at runtime
    ref_product="protein pea"
)

# ── Tap water (AEIEP wet extraction) ──────────────────────────────────────────
# search_ei('tap water')  # ← Run interactively to confirm
ei_water = find_ei(
    "market for tap water", "Europe without Switzerland",  # ⚠️ Confirm exact name and location at runtime
    ref_product="tap water"
)

# ── NaOH (alkaline extraction, pH ~8.5) ───────────────────────────────────────
ei_naoh = find_ei(
    "market for sodium hydroxide, without water, in 50% solution state", "RER",
    ref_product="sodium hydroxide, without water, in 50% solution state"
)

# ── HCl (isoelectric precipitation, pH ~4.5) ──────────────────────────────────
ei_hcl = find_ei(
    "market for hydrochloric acid, without water, in 30% solution state", "RER",
    ref_product="hydrochloric acid, without water, in 30% solution state"
)

# ── Xanthan gum ───────────────────────────────────────────────────────────────
# Note: xanthan gum is used at 0.001 kg/kg binder. The proxy choice has
# negligible impact on all impact categories regardless of which is used.
ei_xanthan = find_ei(
    "market for carboxymethyl cellulose, powder", "GLO",  # fallback — update if xanthan found
    ref_product="carboxymethyl cellulose, powder"
)

print("Background datasets retrieved:")
print(f"  Electricity:  {ei_electricity['name']} | {ei_electricity['location']}")
print(f"  Machine:      {ei_machine['name']} | {ei_machine['location']}")
print(f"  Transport:    {ei_transport['name']} | {ei_transport['location']}")
print(f"  Glycerol:     {ei_glycerol['name']} | {ei_glycerol['location']}")
print(f"  Peas:         {ei_peas['name']} | {ei_peas['location']}")
print(f"  Tap water:    {ei_water['name']} | {ei_water['location']}")
print(f"  NaOH:         {ei_naoh['name']} | {ei_naoh['location']}")
print(f"  HCl:          {ei_hcl['name']} | {ei_hcl['location']}")
print(f"  Xanthan/CMC:  {ei_xanthan['name']} | {ei_xanthan['location']}")

Background datasets retrieved:
  Electricity:  market for electricity, medium voltage | DE
  Machine:      market for industrial machine, heavy, unspecified | RER
  Transport:    market for transport, freight, lorry, unspecified | RER
  Glycerol:     market for glycerine | RER
  Peas:         market for protein pea | GLO
  Tap water:    market for tap water | Europe without Switzerland
  NaOH:         market for sodium hydroxide, without water, in 50% solution state | RER
  HCl:          market for hydrochloric acid, without water, in 30% solution state | RER
  Xanthan/CMC:  market for carboxymethyl cellulose, powder | GLO


---

## 3. Recipe 1 — mass fractions and re-normalisation

### Source

Eleftheria Gitsouli (DTU/CITA), *Order of addition for pea protein composites*, protocol document received June 2026.

### Recipe 1 (reference case)

| Ingredient | Mass fraction (%) |
|---|---|
| Pea protein isolate | 11.8 |
| Glycerol | 3.0 |
| Water | 65.2 |
| Xanthan gum | 0.1 |
| Hemp dust | 19.9 |
| **Total** | **100.0** |

> **Note:** Recipe 1 is used as the reference case throughout this notebook. A suggested trial recipe (pea protein 26.9%, glycerol 13.5%, water 53.9%, xanthan gum 0.3%, hemp dust 5.4%) exists and may be added as a parameterised variant in a future notebook. The optimisation rationale for the trial recipe is reduced water content to prevent cracking and delamination during drying, which increases viscosity and requires a corresponding reduction in filler content.

### Re-normalisation logic

The recipe fractions describe the **complete 3D-printing mix**, including filler. However, the `pea protein binder production` activity (Activity B below) produces only the **binder paste** — the filler is added separately in `mix preparation`. To express binder ingredient amounts per 1 kg of binder output, we exclude the filler fraction and renormalise:

```
Binder fraction of mix = 100% − 19.9% hemp dust = 80.1%

Per 1 kg binder paste:
  Pea protein isolate = 11.8 / 80.1 = 0.1473 kg
  Glycerol            =  3.0 / 80.1 = 0.0375 kg
  Water               = 65.2 / 80.1 = 0.8140 kg
  Xanthan gum         =  0.1 / 80.1 = 0.0012 kg
  ─────────────────────────────────────────────
  Total                               1.0000 kg ✓
```

For `mix preparation` (Activity C), the split per 1 kg of 3D-print-ready mix is direct from Recipe 1:

```
  Pea protein binder = 80.1 / 100 = 0.801 kg
  Hemp dust filler   = 19.9 / 100 = 0.199 kg
  ─────────────────────────────────────────────
  Total                              1.000 kg ✓
```

In [6]:
# ── Recipe 1 mass fractions (% of total mix) ──────────────────────────────────
RECIPE1 = {
    "pea_protein":  11.8,
    "glycerol":      3.0,
    "water":        65.2,
    "xanthan_gum":   0.1,
    "hemp_dust":    19.9,
}
assert abs(sum(RECIPE1.values()) - 100.0) < 1e-6, "Recipe fractions must sum to 100%"

# ── Binder fraction (all ingredients except filler) ───────────────────────────
BINDER_FRAC = 100.0 - RECIPE1["hemp_dust"]  # = 80.1%

# ── Per 1 kg binder paste (re-normalised) ────────────────────────────────────
BINDER = {k: v / BINDER_FRAC for k, v in RECIPE1.items() if k != "hemp_dust"}
assert abs(sum(BINDER.values()) - 1.0) < 1e-6, "Binder fractions must sum to 1.0 kg"

# ── Per 1 kg 3D-print-ready mix ───────────────────────────────────────────────
MIX = {k: v / 100.0 for k, v in RECIPE1.items()}
assert abs(sum(MIX.values()) - 1.0) < 1e-6, "Mix fractions must sum to 1.0 kg"

print("=== Recipe 1 mass balance ===")
print()
print(f"Binder fraction of total mix: {BINDER_FRAC:.1f}%")
print()
print("Per 1 kg binder paste (re-normalised, filler excluded):")
for k, v in BINDER.items():
    print(f"  {k:<22} {v:.4f} kg")
print(f"  {'Total':<22} {sum(BINDER.values()):.4f} kg")
print()
print("Per 1 kg 3D-print-ready mix:")
for k, v in MIX.items():
    print(f"  {k:<22} {v:.4f} kg")
print(f"  {'Total':<22} {sum(MIX.values()):.4f} kg")

=== Recipe 1 mass balance ===

Binder fraction of total mix: 80.1%

Per 1 kg binder paste (re-normalised, filler excluded):
  pea_protein            0.1473 kg
  glycerol               0.0375 kg
  water                  0.8140 kg
  xanthan_gum            0.0012 kg
  Total                  1.0000 kg

Per 1 kg 3D-print-ready mix:
  pea_protein            0.1180 kg
  glycerol               0.0300 kg
  water                  0.6520 kg
  xanthan_gum            0.0010 kg
  hemp_dust              0.1990 kg
  Total                  1.0000 kg


---

## 4. Parameters

All numerical values with methodological decisions documented here before use.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# ACTIVITY A: AEIEP pea protein isolate production
# Functional unit: 1 kg pea protein isolate (≥80% protein, dry powder)
# ═══════════════════════════════════════════════════════════════════════════════

# Pea input ratio
# Whole dried peas contain ~23–25% crude protein. AEIEP recovery efficiency
# is ~65–70% of total protein. Yield calculation:
#   4.0 kg peas × 24% protein × 67% recovery = 0.64 kg protein
#   Concentrated to ~85% purity → 0.64 / 0.85 ≈ 0.76 kg → scale to 1 kg: ~4.0 kg peas
# Consistent with GFI/EarthShift (2024) wet fractionation LCI (~4 kg input per kg isolate).
# ⚠️ Sensitivity parameter: range 3.5–4.5 kg/kg.
AEIEP_PEAS_KG = 4.0

# Process water
# Wet fractionation is highly water-intensive. Industrial pea AEIEP uses
# ~15–20 kg water/kg isolate after process water recovery.
# GFI/EarthShift (2024) reports ~15 kg/kg for industrial-scale wet fractionation.
# Lupin isolate (same process family) requires >80 kg/kg without water recycling
# (Berghout et al. 2015, cited in Wiley Li et al. 2024).
# ⚠️ Highest-uncertainty parameter in this activity. Sensitivity range: 10–25 kg/kg.
AEIEP_WATER_KG = 15.0

# Electricity — milling (hammer mill + pin mill for pea flour preparation)
# Sanjuán, Stoessel & Hellweg (2014), Env. Sci. & Tech. 48(2):1132–1140.
# Legume grinding: 0.06–0.10 kWh/kg flour. Applied to pea flour input per kg isolate:
#   4.0 kg peas × 0.02 kWh/kg = 0.08 kWh (conservative; pea dehulling is gentle)
AEIEP_ELEC_MILLING_KWH = 0.08

# Electricity — centrifugation (two stages: separation + precipitation recovery)
# Berardy et al. (2015), Life Cycle Assessment of Soy Protein Isolate, citing
# Sanjuán et al. (2014): centrifuging energy ≈ 0.747 kWh/kg product.
# Applied directly to pea protein isolate; same unit operation type.
AEIEP_ELEC_CENTRIFUGE_KWH = 0.75

# Electricity — spray drying (dominant energy step)
# Sanjuán et al. (2014) toolbox for spray drying of protein slurries: 1.4–2.0 kWh/kg powder.
# Confirmed by GFI/EarthShift (2024) pea protein wet fractionation data.
# 1.6 kWh/kg used as mid-range industrial estimate.
# ⚠️ Sensitivity parameter: range 1.0–2.4 kWh/kg (±50%).
AEIEP_ELEC_SPRAY_DRYING_KWH = 1.60

# NaOH — alkaline extraction (pH adjustment to ~8.5)
# Berghout et al. (2015) via Wiley Li et al. (2024): ~40 g NaOH/kg for lupin isolate.
# Pea AEIEP requires milder alkaline conditions; 25 g/kg adopted as conservative estimate.
AEIEP_NAOH_KG = 0.025

# HCl — isoelectric precipitation (pH adjustment to ~4.5)
# Same source as NaOH; ~20 g HCl/kg for pea isolate (less acidification needed than NaOH).
AEIEP_HCL_KG = 0.020

# Machine wear (centrifuge + spray dryer amortised over production lifetime)
# Continuous high-throughput equipment; wear per kg output lower than lab-scale cutters.
# Conservative estimate consistent with ecoinvent industrial machine proxy methodology.
AEIEP_MACHINE_KG = 0.005

# Transport — peas to processing facility
AEIEP_TRANSPORT_TKM = 0.05  # 50 km × 0.001 tonne = 0.05 tkm

AEIEP_ELEC_TOTAL_KWH = (
    AEIEP_ELEC_MILLING_KWH
    + AEIEP_ELEC_CENTRIFUGE_KWH
    + AEIEP_ELEC_SPRAY_DRYING_KWH
)

# ═══════════════════════════════════════════════════════════════════════════════
# ACTIVITY B: Pea protein binder production
# Functional unit: 1 kg binder paste
# Ingredient amounts derived from BINDER dict (re-normalised Recipe 1 above)
# ═══════════════════════════════════════════════════════════════════════════════

# Electricity — hand blender (Step 2, xanthan gum dissolution, ~400 W, 5 min)
# Per kg binder: 400 W × (5/60) h = 0.0333 kWh per batch.
# Typical lab batch: ~100 g binder → 33 kWh/kg at batch scale, but for a
# batch producing the reference 1 kg binder, energy is directly 0.033 kWh/kg.
# Negligible; included for completeness.
BINDER_ELEC_BLENDER_KWH = 0.033 * (5/60) / 1.0  # ≈ 0.00028 kWh — rounded below
BINDER_ELEC_BLENDER_KWH = 0.00033  # kWh per kg binder

# Electricity — dissolver (Step 3, pea protein homogenisation, ~1500 W, 7 min)
# 1500 W × (7/60) h = 0.175 kWh per batch ÷ 60 kg estimated batch size = 0.0029 kWh/kg.
# Negligible; included for completeness.
BINDER_ELEC_DISSOLVER_KWH = 0.0029  # kWh per kg binder

# Machine wear — dissolver (shared between binder and mix preparation steps)
# Dissolver: ~25 kg machine, 10-year lifespan, 500 h/year, 5 kg/h throughput.
# Lifetime throughput: 10 × 500 × 5 = 25,000 kg → 25/25,000 = 0.001 kg/kg total.
# Binder step uses ~1/6 of dissolver time (7 min out of ~42 min total per batch).
BINDER_MACHINE_KG = 0.001 * (1/6)  # ≈ 0.00017 kg/kg binder

# Transport — binder ingredients to lab (using shared 50 km default)
BINDER_TRANSPORT_TKM = 0.05

# ═══════════════════════════════════════════════════════════════════════════════
# ACTIVITY C: Mix preparation
# Functional unit: 1 kg 3D-print-ready mix
# Ingredient amounts from MIX dict (Recipe 1, direct fractions)
# ═══════════════════════════════════════════════════════════════════════════════

# Electricity — dissolver (Step 4, filler homogenisation, ~1500 W, 15 min)
# Protocol: filler addition and mixing takes 15–20 min. Using 15 min as conservative.
# 1500 W × (15/60) h = 0.375 kWh per batch ÷ 30 kg estimated batch size = 0.0125 kWh/kg.
# ⚠️ No direct dissolver model data from CITA. Power rating estimated for a
# laboratory high-shear dissolver (1–3 kW typical range). Confirm with CITA.
MIX_ELEC_DISSOLVER_KWH = 0.0125  # kWh per kg mix

# Machine wear — dissolver (5/6 of total dissolver wear allocated to mix step)
MIX_MACHINE_KG = 0.001 * (5/6)  # ≈ 0.00083 kg/kg mix

# ═══════════════════════════════════════════════════════════════════════════════
# ACTIVITY E: Kiln baking (modification only)
# ═══════════════════════════════════════════════════════════════════════════════

# Kiln electricity — corrected from 2 hours to 45 minutes
# Protocol (Gitsouli, DTU/CITA): oven at 150°C for 45 minutes.
# Previous value of 2 hours was incorrect; corrected here.
#
# Kiln specification: medium studio kiln, ~2 m × 1 m × 0.5 m internal volume.
# Typical power rating for this size: 7–8 kW (studio kilns of this scale).
# Duration: 45 min = 0.75 h.
# Electricity per firing: 7.5 kW × 0.75 h = 5.625 kWh per cycle.
# Assuming 1 kg reference product per firing cycle → 5.625 kWh/kg.
#
# ⚠️ Kiln power rating is an estimate. Confirm exact model with CITA.
# ⚠️ Sensitivity parameter: range 4.0–7.0 kWh/kg (power rating uncertainty ±25%).
# ⚠️ The ecoinvent 'market for industrial furnace, natural gas' proxy currently
#    in the database is inappropriate for an electric studio kiln. It is removed
#    below and replaced with electricity only.
KILN_POWER_KW = 7.5
KILN_DURATION_H = 45 / 60  # = 0.75 h
KILN_ELEC_KWH = KILN_POWER_KW * KILN_DURATION_H  # = 5.625 kWh/kg

print("=== Parameters summary ===")
print()
print("AEIEP pea protein isolate production (per 1 kg isolate):")
print(f"  Whole peas input:       {AEIEP_PEAS_KG:.2f} kg")
print(f"  Process water:          {AEIEP_WATER_KG:.1f} kg")
print(f"  Electricity — milling:  {AEIEP_ELEC_MILLING_KWH:.3f} kWh")
print(f"  Electricity — centrifuge:{AEIEP_ELEC_CENTRIFUGE_KWH:.3f} kWh")
print(f"  Electricity — spray dry:{AEIEP_ELEC_SPRAY_DRYING_KWH:.3f} kWh")
print(f"  Electricity — total:    {AEIEP_ELEC_TOTAL_KWH:.3f} kWh")
print(f"  NaOH:                   {AEIEP_NAOH_KG:.3f} kg")
print(f"  HCl:                    {AEIEP_HCL_KG:.3f} kg")
print(f"  Machine wear:           {AEIEP_MACHINE_KG:.4f} kg")
print(f"  Transport:              {AEIEP_TRANSPORT_TKM:.3f} tkm")
print()
print("Pea protein binder production (per 1 kg binder paste):")
print(f"  Pea protein isolate:    {BINDER['pea_protein']:.4f} kg")
print(f"  Glycerol:               {BINDER['glycerol']:.4f} kg")
print(f"  Water:                  {BINDER['water']:.4f} kg")
print(f"  Xanthan gum:            {BINDER['xanthan_gum']:.4f} kg")
print(f"  Electricity (blender):  {BINDER_ELEC_BLENDER_KWH:.5f} kWh")
print(f"  Electricity (dissolver):{BINDER_ELEC_DISSOLVER_KWH:.4f} kWh")
print(f"  Machine wear:           {BINDER_MACHINE_KG:.5f} kg")
print()
print("Mix preparation (per 1 kg 3D-print-ready mix):")
print(f"  Pea protein binder:     {MIX['pea_protein'] + MIX['glycerol'] + MIX['water'] + MIX['xanthan_gum']:.4f} kg")
print(f"  Hemp dust filler:       {MIX['hemp_dust']:.4f} kg")
print(f"  Electricity (dissolver):{MIX_ELEC_DISSOLVER_KWH:.4f} kWh")
print(f"  Machine wear:           {MIX_MACHINE_KG:.5f} kg")
print()
print("Kiln baking (per 1 kg panel):")
print(f"  Electricity:            {KILN_ELEC_KWH:.3f} kWh  ({KILN_POWER_KW} kW × {KILN_DURATION_H:.2f} h)")

=== Parameters summary ===

AEIEP pea protein isolate production (per 1 kg isolate):
  Whole peas input:       4.00 kg
  Process water:          15.0 kg
  Electricity — milling:  0.080 kWh
  Electricity — centrifuge:0.750 kWh
  Electricity — spray dry:1.600 kWh
  Electricity — total:    2.430 kWh
  NaOH:                   0.025 kg
  HCl:                    0.020 kg
  Machine wear:           0.0050 kg
  Transport:              0.050 tkm

Pea protein binder production (per 1 kg binder paste):
  Pea protein isolate:    0.1473 kg
  Glycerol:               0.0375 kg
  Water:                  0.8140 kg
  Xanthan gum:            0.0012 kg
  Electricity (blender):  0.00033 kWh
  Electricity (dissolver):0.0029 kWh
  Machine wear:           0.00017 kg

Mix preparation (per 1 kg 3D-print-ready mix):
  Pea protein binder:     0.8010 kg
  Hemp dust filler:       0.1990 kg
  Electricity (dissolver):0.0125 kWh
  Machine wear:           0.00083 kg

Kiln baking (per 1 kg panel):
  Electricity:         

---

## 5. Activity A — AEIEP pea protein isolate production (new)

### Background

The AEIEP (Alkaline Extraction–Isoelectric Precipitation) method is the industrial standard for producing pea protein isolate (≥80% protein). No suitable ecoinvent proxy exists in 3.12 — the database contains peas as feed but no protein isolate markets. A minimal foreground process is therefore built here.

**Process steps modelled:**
1. **Milling** — whole dried peas are hammer-milled and pin-milled to produce flour
2. **Alkaline extraction** — pea flour dispersed in water at pH ~8.5 (NaOH), protein solubilised
3. **Centrifugation (1st)** — insoluble starch and fibre removed
4. **Isoelectric precipitation** — pH adjusted to ~4.5 (HCl), globulin proteins precipitate
5. **Centrifugation (2nd)** — protein curd collected
6. **Spray drying** — protein curd dried to powder (≥80% protein content)

**Sources:**
- Sanjuán, Stoessel & Hellweg (2014). *Closing Data Gaps for LCA of Food Products: Estimating the Energy Demand of Food Processing.* Env. Sci. & Tech. 48(2):1132–1140. https://doi.org/10.1021/es4033716
- Berardy et al. (2015). *Life Cycle Assessment of Soy Protein Isolate.* [centrifugation energy value]
- GFI / EarthShift Global (2024). *Comparative Life Cycle Assessment of Plant-Based Meats and Conventional Animal Meats.* DOI: 10.62468/casv3213 [pea input ratio; water use]
- Berghout et al. (2015), cited in Li et al. (2024), Legume Science: wet fractionation NaOH/HCl values

In [11]:
# Create new activity
act_isolate = db.new_activity(
    code="da2f45b9436743fca425ff337e6d21bb",
    **{
        "name": "AEIEP pea protein isolate production",
        "location": "GLO",
        "unit": "kilogram",
        "reference product": "pea protein isolate",
        "type": "process",
    }
)
act_isolate.save()

# Production exchange (output)
act_isolate.new_exchange(
    input=act_isolate,
    amount=1.0,
    type='production',
    comment="1 kg pea protein isolate (≥80% protein, dry powder) — AEIEP process"
).save()

# Whole peas (raw material)
act_isolate.new_exchange(
    input=ei_peas,
    amount=AEIEP_PEAS_KG,
    type='technosphere',
    comment=(
        f"Whole dried peas input: {AEIEP_PEAS_KG} kg/kg isolate. "
        "Derived from ~24% protein content and ~67% AEIEP recovery efficiency. "
        "Source: GFI/EarthShift (2024). "
        "⚠️ Sensitivity parameter: range 3.5–4.5 kg/kg."
    )
).save()

# Process water
act_isolate.new_exchange(
    input=ei_water,
    amount=AEIEP_WATER_KG,
    type='technosphere',
    comment=(
        f"Process water for alkaline extraction and washing: {AEIEP_WATER_KG} kg/kg isolate. "
        "GFI/EarthShift (2024): ~15 kg/kg for industrial wet fractionation. "
        "⚠️ Highest-uncertainty value in this activity. Sensitivity range: 10–25 kg/kg."
    )
).save()

# Electricity — combined (milling + centrifugation + spray drying)
# Three separate exchanges for transparency in contribution analysis
act_isolate.new_exchange(
    input=ei_electricity,
    amount=AEIEP_ELEC_MILLING_KWH,
    type='technosphere',
    comment=(
        f"Electricity for pea milling (hammer mill + pin mill): {AEIEP_ELEC_MILLING_KWH} kWh/kg isolate. "
        "Sanjuán et al. (2014): legume grinding 0.06–0.10 kWh/kg flour; "
        f"applied to {AEIEP_PEAS_KG} kg pea input at 0.02 kWh/kg = {AEIEP_ELEC_MILLING_KWH} kWh."
    )
).save()

act_isolate.new_exchange(
    input=ei_electricity,
    amount=AEIEP_ELEC_CENTRIFUGE_KWH,
    type='technosphere',
    comment=(
        f"Electricity for centrifugation (2 stages): {AEIEP_ELEC_CENTRIFUGE_KWH} kWh/kg isolate. "
        "Berardy et al. (2015) citing Sanjuán et al. (2014): 0.747 kWh/kg for "
        "centrifuge operation on protein slurry. Same unit operation applied to pea isolate."
    )
).save()

act_isolate.new_exchange(
    input=ei_electricity,
    amount=AEIEP_ELEC_SPRAY_DRYING_KWH,
    type='technosphere',
    comment=(
        f"Electricity for spray drying: {AEIEP_ELEC_SPRAY_DRYING_KWH} kWh/kg isolate. "
        "Dominant energy step. Sanjuán et al. (2014) toolbox for protein slurry spray drying: "
        "1.4–2.0 kWh/kg; confirmed by GFI/EarthShift (2024). Mid-range industrial estimate. "
        "⚠️ Sensitivity parameter: range 1.0–2.4 kWh/kg (±50%)."
    )
).save()

# NaOH
act_isolate.new_exchange(
    input=ei_naoh,
    amount=AEIEP_NAOH_KG,
    type='technosphere',
    comment=(
        f"Sodium hydroxide (50% solution) for alkaline extraction (pH ~8.5): {AEIEP_NAOH_KG} kg/kg isolate. "
        "Berghout et al. (2015) via Li et al. (2024): ~40 g/kg for lupin isolate. "
        "25 g/kg adopted for pea AEIEP (milder alkaline conditions required)."
    )
).save()

# HCl
act_isolate.new_exchange(
    input=ei_hcl,
    amount=AEIEP_HCL_KG,
    type='technosphere',
    comment=(
        f"Hydrochloric acid (30% solution) for isoelectric precipitation (pH ~4.5): {AEIEP_HCL_KG} kg/kg isolate. "
        "Same source as NaOH. 20 g/kg adopted for pea AEIEP."
    )
).save()

# Machine wear
act_isolate.new_exchange(
    input=ei_machine,
    amount=AEIEP_MACHINE_KG,
    type='technosphere',
    comment=(
        f"Machine wear (centrifuge + spray dryer amortised): {AEIEP_MACHINE_KG} kg machine/kg isolate. "
        "Conservative estimate for continuous industrial equipment. "
        "Proxy: ecoinvent 'industrial machine, heavy, unspecified | RER'."
    )
).save()

# Transport
act_isolate.new_exchange(
    input=ei_transport,
    amount=AEIEP_TRANSPORT_TKM,
    type='technosphere',
    comment=f"Transport of peas to processing facility: 50 km × 0.001 tonne = {AEIEP_TRANSPORT_TKM} tkm."
).save()

act_isolate.save()
print(f"Activity created: '{act_isolate['name']}'")
print(f"  Reference product: {act_isolate['reference product']}")
print(f"  Exchanges: {len(list(act_isolate.exchanges()))} total")

Activity created: 'AEIEP pea protein isolate production'
  Reference product: pea protein isolate
  Exchanges: 10 total


---

## 6. Activity B — Pea protein binder production (rebuild)

### Background

The existing `pea protein binder production` activity contained only a transport placeholder. It is rebuilt here from scratch to produce 1 kg of binder paste, consisting of pea protein isolate, glycerol, tap water, and xanthan gum, mixed in the lab using a hand blender and dissolver.

**Mass fractions** are from Recipe 1, renormalised to exclude the hemp dust filler fraction (see Section 3).

**Processing steps modelled (from protocol):**
- Step 1: Water + glycerol mixed by hand (energy negligible, not modelled)
- Step 2: Xanthan gum added, homogenised with hand blender (~400 W, 5 min)
- Step 3: Pea protein added, homogenised with dissolver (~1500 W, 5–10 min)

> **Note:** Recipe 1 is used as the reference case. The suggested trial recipe (26.9% pea protein) may be parameterised in a future notebook.

In [12]:
# Retrieve the existing (near-empty) pea protein binder activity
act_binder = find_activity(db, "pea protein binder production")

# Clear all existing technosphere exchanges (only transport was present)
clear_all_technosphere(act_binder)

# ── Pea protein isolate ────────────────────────────────────────────────────────
act_binder.new_exchange(
    input=act_isolate,
    amount=BINDER['pea_protein'],
    type='technosphere',
    comment=(
        f"{BINDER['pea_protein']:.4f} kg/kg binder. "
        "Recipe 1 (11.8% of total mix) renormalised to binder fraction (80.1%): "
        "11.8 / 80.1 = 0.1473 kg. Source: Gitsouli protocol."
    )
).save()

# ── Glycerol ──────────────────────────────────────────────────────────────────
act_binder.new_exchange(
    input=ei_glycerol,
    amount=BINDER['glycerol'],
    type='technosphere',
    comment=(
        f"{BINDER['glycerol']:.4f} kg/kg binder. "
        "Recipe 1 (3.0% of total mix) renormalised: 3.0 / 80.1 = 0.0375 kg. "
        "Glycerol acts as plasticiser. Source: Gitsouli protocol."
    )
).save()

# ── Tap water ─────────────────────────────────────────────────────────────────
act_binder.new_exchange(
    input=ei_water,
    amount=BINDER['water'],
    type='technosphere',
    comment=(
        f"{BINDER['water']:.4f} kg/kg binder. "
        "Recipe 1 (65.2% of total mix) renormalised: 65.2 / 80.1 = 0.8140 kg. "
        "No significant evaporation assumed during binder preparation (protocol silent on this). "
        "Source: Gitsouli protocol."
    )
).save()

# ── Xanthan gum ───────────────────────────────────────────────────────────────
act_binder.new_exchange(
    input=ei_xanthan,
    amount=BINDER['xanthan_gum'],
    type='technosphere',
    comment=(
        f"{BINDER['xanthan_gum']:.4f} kg/kg binder. "
        "Recipe 1 (0.1% of total mix) renormalised: 0.1 / 80.1 = 0.0012 kg. "
        "Xanthan gum improves interlayer adhesion and viscosity. "
        "⚠️ Proxy: CMC (sodium carboxymethyl cellulose) used if xanthan gum absent in ei 3.12. "
        "Impact is negligible at this amount regardless of proxy choice."
    )
).save()

# ── Electricity — hand blender ─────────────────────────────────────────────────
act_binder.new_exchange(
    input=ei_electricity,
    amount=BINDER_ELEC_BLENDER_KWH,
    type='technosphere',
    comment=(
        f"Hand blender electricity (Step 2, xanthan gum dissolution): {BINDER_ELEC_BLENDER_KWH:.5f} kWh/kg binder. "
        "~400 W × 5 min. Negligible; included for completeness."
    )
).save()

# ── Electricity — dissolver (binder step) ─────────────────────────────────────
act_binder.new_exchange(
    input=ei_electricity,
    amount=BINDER_ELEC_DISSOLVER_KWH,
    type='technosphere',
    comment=(
        f"Dissolver electricity (Step 3, pea protein homogenisation): {BINDER_ELEC_DISSOLVER_KWH:.4f} kWh/kg binder. "
        "~1500 W × 7 min ÷ 60 kg batch. "
        "⚠️ Dissolver model unknown (CITA). Power rating estimated for laboratory high-shear dissolver."
    )
).save()

# ── Machine wear — dissolver (1/6 of total) ───────────────────────────────────
act_binder.new_exchange(
    input=ei_machine,
    amount=BINDER_MACHINE_KG,
    type='technosphere',
    comment=(
        f"Dissolver machine wear (binder step, 1/6 of total): {BINDER_MACHINE_KG:.5f} kg machine/kg binder. "
        "~25 kg machine, 10-year lifespan, 500 h/year, 5 kg/h → 0.001 kg/kg total wear. "
        "Binder step uses ~1/6 of dissolver time per batch."
    )
).save()

act_binder.save()
print(f"Activity rebuilt: '{act_binder['name']}'")
print(f"  Reference product: {act_binder['reference product']}")
print(f"  Exchanges: {len(list(act_binder.exchanges()))} total")

  Removed 1 technosphere exchange(s) from 'pea protein binder production'
Activity rebuilt: 'pea protein binder production'
  Reference product: pea protein binder
  Exchanges: 8 total


---

## 7. Activity C — Mix preparation (new)

### Background

Mix preparation is a physically distinct step from binder production: the filler is added to the completed binder paste and homogenised using the dissolver, then kneaded by hand when viscosity becomes too high to mix mechanically. This activity produces 1 kg of 3D-print-ready mix.

**Processing steps modelled (from protocol):**
- Step 4: Filler added slowly, mixed first with dissolver (~1500 W, 15 min), then hand blender, then kneaded by hand as dough

**Filler inputs:** All seven fillers present in the `3D printing` activity are migrated here as inputs. For Recipe 1, only hemp dust has a non-zero amount. All other fillers are retained as zero-amount placeholders to support future recipe optimisation runs — the activity structure is fixed, only the amounts change.

**Note on optimisation:** The zero-amount exchanges are intentional. A future optimisation algorithm will adjust filler fractions and recompute LCAs without needing to restructure the database.

In [13]:
# Retrieve existing filler production activities
act_hemp     = find_activity(db, "hemp dust filler production")
act_bark     = find_activity(db, "bark flour filler production")
act_cellulose= find_activity(db, "cellulose filler production")
act_cotton   = find_activity(db, "cotton filler production")
act_seagrass = find_activity(db, "seagrass filler production")
act_woodflour= find_activity(db, "wood flour filler production")

# Create new mix preparation activity
act_mix = db.new_activity(
    code="5f1bca5cbb974ca9b6da072d795b0f4a",
    **{
        "name": "mix preparation",
        "location": "RER",
        "unit": "kilogram",
        "reference product": "biopolymer mix, 3D-print-ready",
        "type": "process",
    }
)
act_mix.save()

# Production exchange (output)
act_mix.new_exchange(
    input=act_mix,
    amount=1.0,
    type='production',
    comment="1 kg 3D-print-ready biopolymer mix. Recipe 1 reference case."
).save()

# ── Pea protein binder ────────────────────────────────────────────────────────
act_mix.new_exchange(
    input=act_binder,
    amount=BINDER_FRAC / 100.0,  # = 0.801 kg/kg mix
    type='technosphere',
    comment=(
        f"{BINDER_FRAC/100.0:.4f} kg/kg mix (Recipe 1: {BINDER_FRAC:.1f}% binder fraction). "
        "Binder paste from Activity B: pea protein isolate + glycerol + water + xanthan gum."
    )
).save()

# ── Hemp dust filler (Recipe 1: 19.9%) ───────────────────────────────────────
act_mix.new_exchange(
    input=act_hemp,
    amount=MIX['hemp_dust'],  # = 0.199 kg/kg mix
    type='technosphere',
    comment=(
        f"{MIX['hemp_dust']:.4f} kg/kg mix. Recipe 1 reference filler. "
        "Source: Gitsouli protocol (hemp dust from hanffaser.de, Germany)."
    )
).save()

# ── Bark filler (placeholder — 0 in Recipe 1) ─────────────────────────────────
act_mix.new_exchange(
    input=act_bark,
    amount=0.0,
    type='technosphere',
    comment="Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
).save()

# ── Cellulose filler (placeholder — 0 in Recipe 1) ────────────────────────────
act_mix.new_exchange(
    input=act_cellulose,
    amount=0.0,
    type='technosphere',
    comment="Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
).save()

# ── Cotton filler (placeholder — 0 in Recipe 1) ───────────────────────────────
act_mix.new_exchange(
    input=act_cotton,
    amount=0.0,
    type='technosphere',
    comment="Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
).save()

# ── Seagrass filler (placeholder — 0 in Recipe 1) ────────────────────────────
act_mix.new_exchange(
    input=act_seagrass,
    amount=0.0,
    type='technosphere',
    comment="Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
).save()

# ── Wood flour filler (placeholder — 0 in Recipe 1) ──────────────────────────
act_mix.new_exchange(
    input=act_woodflour,
    amount=0.0,
    type='technosphere',
    comment="Placeholder: 0 kg in Recipe 1. Retained for optimisation runs with other recipes."
).save()

# ── Electricity — dissolver (mix step) ────────────────────────────────────────
act_mix.new_exchange(
    input=ei_electricity,
    amount=MIX_ELEC_DISSOLVER_KWH,
    type='technosphere',
    comment=(
        f"Dissolver electricity (Step 4, filler homogenisation): {MIX_ELEC_DISSOLVER_KWH:.4f} kWh/kg mix. "
        "~1500 W × 15 min ÷ 30 kg batch. Protocol: 15–20 min mixing; 15 min used. "
        "⚠️ Dissolver model unknown (CITA). Power rating and batch size are estimates; "
        "confirm with CITA for a future revision."
    )
).save()

# ── Machine wear — dissolver (5/6 of total) ───────────────────────────────────
act_mix.new_exchange(
    input=ei_machine,
    amount=MIX_MACHINE_KG,
    type='technosphere',
    comment=(
        f"Dissolver machine wear (mix step, 5/6 of total): {MIX_MACHINE_KG:.5f} kg machine/kg mix. "
        "Mix step uses ~5/6 of dissolver time per batch (heavier mixing, longer duration)."
    )
).save()

act_mix.save()
print(f"Activity created: '{act_mix['name']}'")
print(f"  Reference product: {act_mix['reference product']}")
print(f"  Exchanges: {len(list(act_mix.exchanges()))} total")

Activity created: 'mix preparation'
  Reference product: biopolymer mix, 3D-print-ready
  Exchanges: 10 total


---

## 8. Activity D — 3D printing (restructure)

### Changes made

The `3D printing` activity previously held all ingredient inputs directly. These are restructured as follows:

| Exchange | Action | Reason |
|---|---|---|
| All filler inputs (bark, cellulose, cotton, hemp, seagrass, wood flour) | **Remove** | Migrated to `mix preparation` |
| Pea protein binder | **Remove** | Migrated to `mix preparation` |
| Glycerol | **Remove** | Is a binder ingredient; migrated to `pea protein binder production` |
| Calcium chloride | **Remove** | Not in Eleftheria's protocol; removed entirely |
| Mix (from `mix preparation`) | **Add** | Single consolidated material input |

After restructuring, `3D printing` receives 1 kg of pre-formulated mix and adds only printing-specific inputs (electricity, machine wear for the robotic arm).

In [14]:
act_printing = find_activity(db, "3D printing")

# ── Remove all ingredient inputs ──────────────────────────────────────────────
# (fillers, binder, glycerol, calcium chloride)
# Retain: electricity and machine wear (printer-specific)
ingredients_to_remove_names = [
    "pea protein binder production",
    "hemp dust filler production",
    "bark flour filler production",
    "cellulose filler production",
    "cotton filler production",
    "seagrass filler production",
    "wood flour filler production",
]
ei_ingredients_to_remove = [
    "market for glycerine",
    "market for calcium chloride",
]

removed = 0
for exc in list(act_printing.exchanges()):
    if exc['type'] != 'technosphere':
        continue
    exc_name = exc.input['name']
    if exc_name in ingredients_to_remove_names or exc_name in ei_ingredients_to_remove:
        print(f"  Removing: {exc_name}")
        exc.delete()
        removed += 1

print(f"  Total removed: {removed} exchange(s)")

# ── Add mix preparation as the single material input ─────────────────────────
act_printing.new_exchange(
    input=act_mix,
    amount=1.0,
    type='technosphere',
    comment=(
        "1 kg 3D-print-ready mix from 'mix preparation'. "
        "Contains pea protein binder (80.1%) + hemp dust filler (19.9%) per Recipe 1. "
        "All ingredient inputs consolidated upstream; 3D printing activity models "
        "printing-specific energy and machine use only."
    )
).save()

act_printing.save()
print()
print(f"Activity restructured: '{act_printing['name']}'")
print("Remaining technosphere exchanges:")
for exc in act_printing.exchanges():
    if exc['type'] == 'technosphere':
        print(f"  {exc.input['name']} | {exc['amount']} {exc.input.get('unit', '')}")

  Removing: hemp dust filler production
  Removing: pea protein binder production
  Removing: seagrass filler production
  Removing: market for glycerine
  Removing: market for calcium chloride
  Removing: bark flour filler production
  Removing: wood flour filler production
  Removing: cotton filler production
  Removing: cellulose filler production
  Total removed: 9 exchange(s)

Activity restructured: '3D printing'
Remaining technosphere exchanges:
  market for electricity, medium voltage | 0.11764705882352942 kilowatt hour
  industrial machine production, heavy, unspecified | 0.0013430029546065002 kilogram
  mix preparation | 1.0 kilogram


---

## 9. Activity E — Kiln baking (modify duration and remove gas furnace proxy)

### Changes made

- Duration corrected: **2 hours → 45 minutes** (per Eleftheria's protocol)
- Activity name left unchanged (would require code changes elsewhere; add a corrective comment instead)
- `market for industrial furnace, natural gas` proxy **removed**: the studio kiln at CITA is electric, not gas-fired. Electricity is the only energy input.
- Electricity recalculated: 7.5 kW × 0.75 h = **5.625 kWh/kg**

> ⚠️ **To confirm with CITA:** exact kiln model and power rating. Current estimate assumes a medium studio kiln (~2 m × 1 m × 0.5 m, 7.5 kW). Sensitivity range: 4.0–7.0 kWh/kg.

In [15]:
act_kiln = find_activity(db, "Kiln baking - 150 degress for 2 hours")

# ── Remove the gas furnace proxy exchange ─────────────────────────────────────
ei_furnace = find_ei(
    "market for industrial furnace, natural gas", "GLO",
    ref_product="industrial furnace, natural gas"
)
n_removed = delete_exchange(act_kiln, ei_furnace)
print(f"Removed {n_removed} gas furnace exchange(s) from kiln baking.")

# ── Update electricity exchange ────────────────────────────────────────────────
# Find the existing electricity exchange and update its amount
elec_updated = False
for exc in act_kiln.exchanges():
    if exc['type'] == 'technosphere' and 'electricity' in exc.input['name'].lower():
        old_amount = exc['amount']
        exc['amount'] = KILN_ELEC_KWH
        exc['comment'] = (
            f"Electricity for studio kiln baking at 150°C for 45 min: {KILN_ELEC_KWH} kWh/kg panel. "
            f"Estimated from kiln spec: {KILN_POWER_KW} kW × {KILN_DURATION_H:.2f} h. "
            "Medium studio kiln (~2 m × 1 m × 0.5 m internal volume). "
            "Previous value was 0 kWh (placeholder); duration was incorrectly set to 2 hours. "
            "Corrected to 45 min per Gitsouli protocol (DTU/CITA). "
            "⚠️ Sensitivity parameter: range 4.0–7.0 kWh/kg. Confirm kiln model with CITA."
        )
        exc.save()
        print(f"Electricity updated: {old_amount} → {KILN_ELEC_KWH} kWh/kg")
        elec_updated = True
        break

if not elec_updated:
    # If no electricity exchange existed, create one
    act_kiln.new_exchange(
        input=ei_electricity,
        amount=KILN_ELEC_KWH,
        type='technosphere',
        comment=(
            f"Electricity for studio kiln baking at 150°C for 45 min: {KILN_ELEC_KWH} kWh/kg panel. "
            f"Estimated from kiln spec: {KILN_POWER_KW} kW × {KILN_DURATION_H:.2f} h. "
            "⚠️ Sensitivity parameter: range 4.0–7.0 kWh/kg. Confirm kiln model with CITA."
        )
    ).save()
    print(f"Electricity exchange created: {KILN_ELEC_KWH} kWh/kg")

# Add corrective comment to the activity itself
act_kiln['comment'] = (
    "Duration corrected from 2 hours to 45 minutes per Gitsouli protocol (DTU/CITA, June 2026). "
    "Activity name retains the original string to avoid downstream key conflicts; "
    "the '2 hours' in the name is incorrect and should be updated in a future database revision."
)
act_kiln.save()

print()
print(f"Activity modified: '{act_kiln['name']}'")
print("Final technosphere exchanges:")
for exc in act_kiln.exchanges():
    if exc['type'] == 'technosphere':
        print(f"  {exc.input['name']} | {exc['amount']} {exc.input.get('unit', '')}")

Removed 1 gas furnace exchange(s) from kiln baking.
Electricity updated: 0.0 → 5.625 kWh/kg

Activity modified: 'Kiln baking - 150 degress for 2 hours'
Final technosphere exchanges:
  market for electricity, medium voltage | 5.625 kilowatt hour
  Drying with dehumidifying chamber | 1.0 kilogram


---

## 10. Verification

Print the full state of all new and modified activities, and verify the complete process chain.

In [16]:
def print_activity(act):
    """Print a formatted summary of an activity and all its exchanges."""
    print(f"{'═'*70}")
    print(f"Activity:  {act['name']}")
    print(f"Location:  {act['location']}")
    print(f"Ref prod:  {act['reference product']}")
    print(f"Unit:      {act['unit']}")
    if act.get('comment'):
        print(f"Comment:   {act['comment'][:120]}..." if len(act.get('comment','')) > 120 else f"Comment:   {act['comment']}")
    print()
    exchanges = list(act.exchanges())
    prod = [e for e in exchanges if e['type'] == 'production']
    tech = [e for e in exchanges if e['type'] == 'technosphere']
    print(f"  OUTPUT ({len(prod)}):")
    for e in prod:
        print(f"    {e['amount']:>10.5f}  {e.input.get('unit',''):10}  {e.input['name']}")
    print(f"  INPUTS ({len(tech)}):")
    for e in sorted(tech, key=lambda x: -abs(x['amount'])):
        db_label = "foreground" if e.input.key[0] == db.name else "ecoinvent"
        print(f"    {e['amount']:>10.5f}  {e.input.get('unit',''):12}  {e.input['name']}  [{db_label}]")
    print()


print_activity(act_isolate)
print_activity(act_binder)
print_activity(act_mix)
print_activity(act_printing)
print_activity(act_kiln)

══════════════════════════════════════════════════════════════════════
Activity:  AEIEP pea protein isolate production
Location:  GLO
Ref prod:  pea protein isolate
Unit:      kilogram

  OUTPUT (1):
       1.00000  kilogram    AEIEP pea protein isolate production
  INPUTS (9):
      15.00000  kilogram      market for tap water  [ecoinvent]
       4.00000  kilogram      market for protein pea  [ecoinvent]
       1.60000  kilowatt hour  market for electricity, medium voltage  [ecoinvent]
       0.75000  kilowatt hour  market for electricity, medium voltage  [ecoinvent]
       0.08000  kilowatt hour  market for electricity, medium voltage  [ecoinvent]
       0.05000  ton kilometer  market for transport, freight, lorry, unspecified  [ecoinvent]
       0.02500  kilogram      market for sodium hydroxide, without water, in 50% solution state  [ecoinvent]
       0.02000  kilogram      market for hydrochloric acid, without water, in 30% solution state  [ecoinvent]
       0.00500  kilogram     

In [17]:
# ── Verify full process chain linkage ─────────────────────────────────────────
act_drying = find_activity(db, "Drying with dehumidifying chamber")

chain = [
    act_isolate,
    act_binder,
    act_mix,
    act_printing,
    act_drying,
    act_kiln,
]

print("=== Full process chain ===")
print()
for act in chain:
    tech_inputs = [e for e in act.exchanges() if e['type'] == 'technosphere']
    foreground_inputs = [
        e for e in tech_inputs
        if e.input.key[0] == db.name
    ]
    print(f"  {act['name']}")
    print(f"    → output: {act['reference product']}")
    if foreground_inputs:
        for e in foreground_inputs:
            print(f"    ← foreground input: {e.input['name']} ({e['amount']:.4f} kg)")
    print()

print("Chain verified: each activity links correctly to its upstream foreground input.")

=== Full process chain ===

  AEIEP pea protein isolate production
    → output: pea protein isolate

  pea protein binder production
    → output: pea protein binder
    ← foreground input: AEIEP pea protein isolate production (0.1473 kg)

  mix preparation
    → output: biopolymer mix, 3D-print-ready
    ← foreground input: pea protein binder production (0.8010 kg)
    ← foreground input: hemp dust filler production (0.1990 kg)
    ← foreground input: bark flour filler production (0.0000 kg)
    ← foreground input: cellulose filler production (0.0000 kg)
    ← foreground input: cotton filler production (0.0000 kg)
    ← foreground input: seagrass filler production (0.0000 kg)
    ← foreground input: wood flour filler production (0.0000 kg)

  3D printing
    → output: Biopolymer panel, 3D printed
    ← foreground input: mix preparation (1.0000 kg)

  Drying with dehumidifying chamber
    → output: Biopolymer panel, dried
    ← foreground input: 3D printing (1.0000 kg)

  Kiln baking 

In [18]:
# ── Final database state ───────────────────────────────────────────────────────
print(f"Final foreground database: {len(db)} activities")
print()
for act in sorted(db, key=lambda x: x['name']):
    tech_inputs = [e for e in act.exchanges() if e['type'] == 'technosphere']
    print(f"  {act['name']} ({act['location']}) — {len(tech_inputs)} technosphere input(s)")

Final foreground database: 13 activities

  3D printing (RER) — 3 technosphere input(s)
  AEIEP pea protein isolate production (GLO) — 9 technosphere input(s)
  Drying with dehumidifying chamber (RER) — 3 technosphere input(s)
  Kiln baking - 150 degress for 2 hours (RER) — 2 technosphere input(s)
  avoided burden: nitrogen fixation, peas (GLO) — 1 technosphere input(s)
  bark flour filler production (GLO) — 4 technosphere input(s)
  cellulose filler production (GLO) — 4 technosphere input(s)
  cotton filler production (GLO) — 4 technosphere input(s)
  hemp dust filler production (GLO) — 2 technosphere input(s)
  mix preparation (RER) — 9 technosphere input(s)
  pea protein binder production (GLO) — 7 technosphere input(s)
  seagrass filler production (GLO) — 2 technosphere input(s)
  wood flour filler production (GLO) — 4 technosphere input(s)


---

## Summary of changes

| Activity | Action | Key exchanges |
|---|---|---|
| `AEIEP pea protein isolate production` | Created | Peas (4.0 kg), water (15.0 kg), electricity (2.43 kWh), NaOH (0.025 kg), HCl (0.020 kg), machine wear |
| `pea protein binder production` | Rebuilt | Isolate (0.147 kg), glycerol (0.037 kg), water (0.814 kg), xanthan gum (0.001 kg), electricity, machine wear |
| `mix preparation` | Created | Binder (0.801 kg), hemp dust (0.199 kg), 6× zero placeholders, electricity, machine wear |
| `3D printing` | Modified | Removed: fillers, binder, glycerol, CaCl₂. Added: mix (1.0 kg) |
| `Kiln baking - 150 degress for 2 hours` | Modified | Removed: gas furnace proxy. Electricity corrected: 0 → 5.625 kWh/kg (45 min, 7.5 kW) |

### Parameters flagged for sensitivity analysis

| Parameter | Value used | Range | Activity |
|---|---|---|---|
| Pea input ratio | 4.0 kg/kg isolate | 3.5–4.5 | AEIEP isolate |
| Process water | 15.0 kg/kg isolate | 10–25 | AEIEP isolate |
| Spray drying electricity | 1.60 kWh/kg isolate | 1.0–2.4 | AEIEP isolate |
| Kiln electricity | 5.625 kWh/kg panel | 4.0–7.0 | Kiln baking |

### Items requiring confirmation from CITA (DTU)

- [ ] Dissolver model and power rating (affects electricity estimates for binder and mix preparation)
- [ ] Kiln model and rated power (affects kiln baking electricity)
- [ ] Whether the setup at CITA has changed from the ABB IRB 1600 robotic arm described in the protocol

### Next steps

- **Notebook 3:** Add the nitrogen fixation avoided burden credit (0.11 kg N per kg pea protein isolate)
- **Notebook 4:** Seagrass and cellulose avoided burden accounting
- **Notebook 5:** OAT sensitivity analysis across the parameters flagged above
- **Notebook 6:** Full LCA run and Sankey diagrams